In [11]:
from sklearn.datasets import fetch_openml
import numpy as np
import pandas as pd
from sklearn.tree import DecisionTreeRegressor
from sklearn.model_selection import GridSearchCV, train_test_split, RepeatedKFold, cross_validate
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

In [12]:
boston = fetch_openml(name="boston", version=1, as_frame=True)
df = boston.frame
df.head()

,CRIM,ZN,INDUS,CHAS,NOX,RM,AGE,DIS,RAD,TAX,PTRATIO,B,LSTAT,MEDV
0,0.00632,18.0,2.31,0,0.538,6.575,65.2,4.0900,1,296.0,15.3,396.90,4.98,24.0
1,0.02731,0.0,7.07,0,0.469,6.421,78.9,4.9671,2,242.0,17.8,396.90,9.14,21.6
2,0.02729,0.0,7.07,0,0.469,7.185,61.1,4.9671,2,242.0,17.8,392.83,4.03,34.7
3,0.03237,0.0,2.18,0,0.458,6.998,45.8,6.0622,3,222.0,18.7,394.63,2.94,33.4
4,0.06905,0.0,2.18,0,0.458,7.147,54.2,6.0622,3,222.0,18.7,396.90,5.33,36.2


In [13]:
X = df.drop(columns="MEDV")
y = pd.to_numeric(df["MEDV"], errors="coerce")

valid_idx = y.notna()
X = X.loc[valid_idx]
y = y.loc[valid_idx]

In [14]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [15]:
model = DecisionTreeRegressor(random_state=42)

param_grid = {
    "max_depth": [3, 5, 8, 12, None],
    "min_samples_split": [2, 5, 10, 20],
    "min_samples_leaf": [1, 2, 4, 8],
    "max_features": [None, "sqrt", "log2"],
    "ccp_alpha": [0.0, 0.001, 0.01, 0.05]
}

cv = RepeatedKFold(n_splits=5, n_repeats=3, random_state=42)
grid = GridSearchCV(
    estimator=model,
    param_grid=param_grid,
    cv=cv,
    scoring="neg_root_mean_squared_error",
    n_jobs=-1
)

In [16]:
grid.fit(X_train, y_train)

print("Best Parameters:", grid.best_params_)
print("Best CV RMSE:", -grid.best_score_)

Best Parameters: {'ccp_alpha': 0.0, 'max_depth': 5, 'max_features': None, 'min_samples_leaf': 8, 'min_samples_split': 2}
Best CV RMSE: 4.665753201248352


In [17]:
best_model = grid.best_estimator_
y_pred = best_model.predict(X_test)

In [19]:

mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print("Test MAE:", mae)
print("Test R2:", r2)

Test MAE: 2.744852777103274
Test R2: 0.7721688298945275


In [20]:
cv_results = cross_validate(
    best_model,
    X,
    y,
    cv=cv,
    scoring={
        "rmse": "neg_root_mean_squared_error",
        "mae": "neg_mean_absolute_error",
        "r2": "r2",
    },
    n_jobs=-1
)

print("CV RMSE:", -cv_results["test_rmse"].mean())
print("CV MAE:", -cv_results["test_mae"].mean())
print("CV R2:", cv_results["test_r2"].mean())

CV RMSE: 4.502642535589634
CV MAE: 3.0509487139430695
CV R2: 0.7433319376362422


In [21]:
train_r2 = best_model.score(X_train, y_train)
test_r2 = best_model.score(X_test, y_test)

print("Train R2:", train_r2)
print("Test R2:", test_r2)

Train R2: 0.8482641207222953
Test R2: 0.7721688298945275
